# Динамическое ценообразование — MVP
Это демонстрационный ноутбук, использующий централизованную модель из `model/pricing.py`.

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Добавляем корень проекта для импорта модели
sys.path.append(str(Path().absolute().parent))
from model.pricing import *

print("Модели успешно загружены.")

## 1. Исследование данных (EDA)

In [ ]:
data_path = Path().absolute().parent / 'data' / 'sales_history.csv'
df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])

print(f"Загружено строк: {len(df)}")
display(df.groupby('product')[['our_price', 'sales', 'revenue']].mean())

# График зависимости
plt.figure(figsize=(10, 5))
for prod in PRODUCTS:
    sub = df[df['product'] == prod]
    plt.scatter(sub['our_price'], sub['sales'], label=prod, alpha=0.5)
plt.title("Зависимость продаж от цены (все товары)")
plt.xlabel("Цена")
plt.ylabel("Продажи")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Эвристика (Rule-based Pricing)

In [ ]:
results_rules = []
last_date = df['date'].max()
last_day = df[df['date'] == last_date]

for _, row in last_day.iterrows():
    prod = row['product']
    avg7 = df[(df['product'] == prod) & (df['date'] < last_date)].tail(7)['sales'].mean()
    rec_p, rule = apply_rules(row, avg7)
    fc = forecast(prod, rec_p, row['revenue'])
    
    results_rules.append({
        'Товар': prod, 
        'Старая цена': row['our_price'], 
        'Новая цена': rec_p, 
        'Правило': rule,
        'Рост выручки %': fc['growth_pct']
    })

pd.DataFrame(results_rules)

## 3. Регрессионная модель

In [ ]:
plt.figure(figsize=(12, 8))
for i, prod in enumerate(PRODUCTS, 1):
    a, b, opt_p = fit_regression(df, prod)
    
    # Отрисовка кривой выручки
    prices = np.linspace(opt_p*0.5, opt_p*1.5, 100)
    revs = prices * (a - b * prices)
    
    plt.subplot(2, 3, i)
    plt.plot(prices, revs)
    plt.axvline(opt_p, color='g', label='Optimal')
    plt.title(f"{prod}: {opt_p} RUB")
    plt.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 4. Сравнение методов

In [ ]:
comp = []
for prod in PRODUCTS:
    # Эвристика (последний день)
    row = df[(df['product'] == prod)].iloc[-1]
    avg7 = df[df['product'] == prod]['sales'].tail(7).mean()
    p_rule, _ = apply_rules(row, avg7)
    
    # Регрессия
    _, _, p_reg = fit_regression(df, prod)
    
    comp.append({'Товар': prod, 'Текущая': row['our_price'], 'Эвристика': p_rule, 'Регрессия': p_reg})

comp_df = pd.DataFrame(comp)
comp_df.set_index('Товар').plot(kind='bar', figsize=(10, 5))
plt.title("Сравнение рекомендуемых цен")
plt.xticks(rotation=0)
plt.show()

## 5. Симуляция будущего

In [ ]:
sim_df = simulate(df, n_steps=14, method='regression')

plt.figure(figsize=(12, 5))
plt.plot(df.groupby('date')['revenue'].sum(), label='История')
plt.plot(sim_df.groupby('date')['revenue'].sum(), label='Симуляция (Regression)', ls='--')
plt.axvline(df['date'].max(), color='k', alpha=0.3)
plt.title("Прогноз выручки при переходе на оптимальные цены")
plt.legend()
plt.show()

## 6. Итоговый отчет

In [ ]:
print("=== РЕКОМЕНДАЦИИ ПО ПЕРЕОЦЕНКЕ ===")
for prod in PRODUCTS:
    last_row = df[df['product'] == prod].iloc[-1]
    _, _, op = fit_regression(df, prod)
    fc = forecast(prod, op, last_row['revenue'])
    
    print(f"Товар {prod}: старая цена {last_row['our_price']:.2f} руб → новая цена {op:.2f} руб | прогноз выручки: {fc['growth_pct']:+.1f}%")